In [1]:
%pwd

'c:\\Customer_churn_project\\notebooks'

In [2]:
import os

os.chdir("../")

In [3]:
from dataclasses import dataclass
from pathlib import Path

In [4]:

@dataclass(frozen=True)
class ModelTrainingConfig:
    root_dir:Path
    training_data_path:Path
    base_model_path:Path
    tuned_model_path:Path

In [5]:
from src.CustomerChurnPrediction.constants import *
from src.CustomerChurnPrediction.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:

    def __init__(self, config_filepath=CONFIG_FILE_PATH):
        self.config = read_yaml(config_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_training(self):

        config = self.config.model_training

        create_directories([config.root_dir])

        model_training_config = ModelTrainingConfig(
            root_dir=config.root_dir,
            training_data_path=config.training_data_path,
            base_model_path=config.base_model_path,
            tuned_model_path=config.tuned_model_path
        )

        return model_training_config

In [ ]:
# import dagshub
# dagshub.init(repo_owner='udaypatel2209', repo_name='Customer-Churn-Prediction', mlflow=True)

In [ ]:
# import mlflow

# mlflow.set_registry_uri("https://dagshub.com/udaypatel2209/Customer-Churn-Prediction.mlflow")
# mlflow.set_tracking_uri("https://dagshub.com/udaypatel2209/Customer-Churn-Prediction.mlflow")
# mlflow.set_experiment("Telco Churn - Baseline Models")

In [ ]:
import time
import pandas as pd
import optuna
import json
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
import sys
from dotenv import load_dotenv
from sqlalchemy import create_engine
from src.CustomerChurnPrediction.utils.logger import logger
from src.CustomerChurnPrediction.utils.exception import CustomException

load_dotenv()

True

In [ ]:
class ModelTraining:
    """Trains a base XGBoost model and a tuned XGBoost model (via Optuna), saving both to disk."""

    THRESHOLD = 0.25

    def __init__(self, config: ModelTrainingConfig, n_trials: int = 30):
        """Sets up config and trial count."""
        self.config = config
        self.n_trials = n_trials

    def get_training_data(self) -> pd.DataFrame:
        """Loads the transformed training CSV."""
        logger.info(f"Loading training data from: {self.config.training_data_path}")
        return pd.read_csv(self.config.training_data_path)

    def split_data(self, df: pd.DataFrame, target_col: str = "Churn", test_size: float = 0.2):
        """Splits the data into train and test sets."""
        X = df.drop(columns=[target_col])
        y = df[target_col]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        logger.info(f"Split data: {X_train.shape[0]} train rows, {X_test.shape[0]} test rows")
        return X_train, X_test, y_train, y_test

    def get_scale_pos_weight(self, y_train) -> float:
        """Calculates the class weight used to correct for churn-class imbalance."""
        return (y_train == 0).sum() / (y_train == 1).sum()

    @staticmethod
    def save_model(model: XGBClassifier, path: str) -> None:
        """Saves a model to the given path, creating the folder if needed."""
        os.makedirs(os.path.dirname(path), exist_ok=True)
        joblib.dump(model, path)
        logger.info(f"Model saved to {path}")

    @staticmethod
    def save_model_for_github(model: XGBClassifier, filename: str = "tuned_model.pkl") -> None:
        """Saves a model into a small 'model/' folder meant to be committed to GitHub, since artifacts/ is not pushed."""
        github_dir = "model"
        os.makedirs(github_dir, exist_ok=True)
        path = os.path.join(github_dir, filename)
        joblib.dump(model, path)
        logger.info(f"Model saved for GitHub at {path}")

    def train_base_model(self, X_train, y_train) -> XGBClassifier:
        """Trains the base XGBoost model and saves it to disk."""
        scale_pos_weight = self.get_scale_pos_weight(y_train)

        params = {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 6,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "random_state": 42,
            "n_jobs": -1,
            "scale_pos_weight": scale_pos_weight,
            "eval_metric": "logloss",
        }

        start_train = time.time()
        model = XGBClassifier(**params)
        model.fit(X_train, y_train)
        logger.info(f"Base model trained in {time.time() - start_train:.2f} seconds")

        self.save_model(model, self.config.base_model_path)

        return model

    def tune_model(self, X_train, y_train, X_test, y_test) -> XGBClassifier:
        """
        Runs Optuna to find the best hyperparameters, then trains and saves
        the final tuned model. Hyperparameters are evaluated on a VALIDATION
        split carved out of X_train (not on X_test/y_test), so the test set
        never influences which params get chosen.
        """
        # Validation split for the Optuna search only -- carved from X_train.
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )
        val_scale_pos_weight = self.get_scale_pos_weight(y_tr)

        def objective(trial: optuna.Trial) -> float:
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 300, 800),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "gamma": trial.suggest_float("gamma", 0, 5),
                "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
                "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
                "random_state": 42,
                "n_jobs": -1,
                "scale_pos_weight": val_scale_pos_weight,
                "eval_metric": "logloss",
            }

            model = XGBClassifier(**params)
            model.fit(X_tr, y_tr)
            proba = model.predict_proba(X_val)[:, 1]
            y_pred = (proba >= self.THRESHOLD).astype(int)
            return recall_score(y_val, y_pred, pos_label=1)

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=self.n_trials)

        logger.info(f"Best trial: {study.best_trial.number} | Val Recall: {study.best_value:.4f}")
        logger.info(f"Best params: {study.best_params}")

        # Refit on the FULL training set (X_train, not X_tr) using the full
        # scale_pos_weight (not the validation-split one).
        scale_pos_weight = self.get_scale_pos_weight(y_train)
        best_params = {
            **study.best_params,
            "random_state": 42,
            "n_jobs": -1,
            "scale_pos_weight": scale_pos_weight,
            "eval_metric": "logloss",
        }

        start_train = time.time()
        best_model = XGBClassifier(**best_params)
        best_model.fit(X_train, y_train)
        logger.info(f"Tuned model trained in {time.time() - start_train:.2f} seconds")

        self.save_model(best_model, self.config.tuned_model_path)
        self.save_model_for_github(best_model)

        return best_model
    
    @staticmethod
    def save_best_params(best_params:dict):
        with open("artifacts/model/best_params.json","w") as f:
            json.dump(best_params, f, indent=4)

    def run_training(self) -> None:
        """Runs the full training pipeline: load data, split, train base model, tune model."""
        df = self.get_training_data()
        X_train, X_test, y_train, y_test = self.split_data(df)

        self.train_base_model(X_train, y_train)
        self.tune_model(X_train, y_train, X_test, y_test)

In [ ]:
try:
    config = ConfigurationManager()
    model_training_config = config.get_model_training()
    model_training = ModelTraining(model_training_config)
    model_training.run_training()
except Exception as e:
    raise CustomException(e, sys)

[2026-06-22 00:49:29,625] 33 CustomerChurnPrediction - INFO - yaml file: config\config.yml loaded successfully
[2026-06-22 00:49:29,633] 50 CustomerChurnPrediction - INFO - created directory at: artifacts
[2026-06-22 00:49:29,638] 50 CustomerChurnPrediction - INFO - created directory at: artifacts/training
[2026-06-22 00:49:29,643] 13 CustomerChurnPrediction - INFO - Loading training data from: artifacts/data_transformation/transformed_data.csv


[2026-06-22 00:49:29,691] 24 CustomerChurnPrediction - INFO - Split data: 5625 train rows, 1407 test rows
[2026-06-22 00:49:30,746] 66 CustomerChurnPrediction - INFO - Base model trained in 1.05 seconds
[2026-06-22 00:49:30,770] 36 CustomerChurnPrediction - INFO - Model saved to artifacts/model/base_model.pkl


[I 2026-06-22 00:49:30,786] A new study created in memory with name: no-name-b1721a16-1ae7-4630-9c30-a56395bf2bd0
[I 2026-06-22 00:49:31,168] Trial 0 finished with value: 0.919732441471572 and parameters: {'n_estimators': 458, 'learning_rate': 0.08300592857296049, 'max_depth': 5, 'subsample': 0.7464804322675189, 'colsample_bytree': 0.5487101416061597, 'min_child_weight': 3, 'gamma': 4.001451301339855, 'reg_alpha': 4.4963446352244505, 'reg_lambda': 3.7113679800554964}. Best is trial 0 with value: 0.919732441471572.
[I 2026-06-22 00:49:31,737] Trial 1 finished with value: 0.919732441471572 and parameters: {'n_estimators': 731, 'learning_rate': 0.03987605659397595, 'max_depth': 4, 'subsample': 0.9440328705632597, 'colsample_bytree': 0.790709009046296, 'min_child_weight': 7, 'gamma': 1.973517925835206, 'reg_alpha': 0.2022570357037623, 'reg_lambda': 3.5478970395066316}. Best is trial 0 with value: 0.919732441471572.
[I 2026-06-22 00:49:31,994] Trial 2 finished with value: 0.9163879598662207

[2026-06-22 00:49:44,721] 111 CustomerChurnPrediction - INFO - Best trial: 13 | Val Recall: 0.9465
[2026-06-22 00:49:44,723] 112 CustomerChurnPrediction - INFO - Best params: {'n_estimators': 300, 'learning_rate': 0.011204380446487909, 'max_depth': 3, 'subsample': 0.625077047542465, 'colsample_bytree': 0.6498585043262818, 'min_child_weight': 8, 'gamma': 3.0938588854860134, 'reg_alpha': 3.0338945879809387, 'reg_lambda': 1.3130208515127353}
[2026-06-22 00:49:44,991] 128 CustomerChurnPrediction - INFO - Tuned model trained in 0.27 seconds
[2026-06-22 00:49:44,998] 36 CustomerChurnPrediction - INFO - Model saved to artifacts/model/tuned_model.pkl
[2026-06-22 00:49:45,007] 45 CustomerChurnPrediction - INFO - Model saved for GitHub at model\tuned_model.pkl
